## Building a Hierarchical (RAPTOR) Index for Multimodal RAG

This notebook constructs a RAPTOR-style hierarchical index for our collection of 1023 research papers, extending the base RAG pipeline into a multi-level retrieval system. The goal is to move from raw L0 chunks to structured L1/L2 summaries, embed each level, and finally build FAISS indexes that enable efficient hierarchical retrieval for downstream models.

The workflow implemented here includes:
- loading the pre-extracted L0 chunks,
- generating L1 and L2 summary nodes via GPT-4o-mini,
- computing embeddings using a lightweight 384-dim text model,
- exporting all nodes to a consolidated JSONL file, and
- saving FAISS indexes for L1 and L2 levels.

By the end, we obtain a complete RAPTOR hierarchy for every paper, enabling coarse-to-fine retrieval within the larger RAG system.


### Environment Setup and Core Dependencies

In [2]:
import os
import json
import pickle
import numpy as np
import openai
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Configure OpenAI API key
assert 'OPENAI_API_KEY' in os.environ, "Please set the OPENAI_API_KEY environment variable."
openai.api_key = os.getenv('OPENAI_API_KEY')

# Load the embedding model
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Loading Level-0 (L0) Chunks from the RAG Dataset

In this step, we load all base document chunks that form the foundation of the RAPTOR hierarchy. Each JSON file corresponds to a single paper and contains the raw paragraph, figure, table, and equation chunks.

**Key points:**

- **Source directory:** `data2/rag_chunks_v2/`  
  Each file contains all extracted chunks for one paper.

- **L0 node creation:**  
  For every chunk, we generate a node with the ID pattern:  
  `paper_id_L0_<index>`

- **Embedding dimension logic:**  
  - Textual chunks → **384-dim embeddings**  
  - Figure/table chunks → **896-dim embeddings** (from original image+caption embeddings)

- **Metadata stored:**
  - Page number (`min_page` = `max_page`)
  - Empty `children_ids` (L0 nodes do not have children)

- **Data structures:**
  - `all_papers_L0[paper_id]` holds the list of L0 nodes for each paper  
  - `raptor_nodes` collects **all nodes** across the entire dataset

- **Progress tracking:**  
  A tqdm progress bar shows progress across all 1023 papers.

- **Summary statistics:**  
  After loading, we compute how many L1 (groups of 8) and L2 (groups of 6 L1 nodes) summaries will be needed.

This prepares the dataset for hierarchical summarization in the next stage.


In [4]:
from glob import glob

# Directory containing the RAG chunk JSON files
chunk_files = sorted(glob("../data2/rag_chunks_v2/*.json"))  # Sorted for consistency

all_papers_L0 = {}        # Mapping of paper_id -> list of L0 node dicts
raptor_nodes = []         # Master list of all nodes (to be saved later)
total_L0_count = 0

for file_path in tqdm(chunk_files, desc="Loading L0 chunks"):
    paper_id = os.path.basename(file_path).split(".rag.chunks.json")[0]
    with open(file_path, "r") as f:
        chunks = json.load(f)
    # Create L0 nodes for this paper
    L0_nodes = []
    for idx, chunk in enumerate(chunks):
        content = chunk.get("content", "")
        node_id = f"{paper_id}_L0_{idx}"
        # Determine embedding dimension by chunk type
        ctype = chunk.get("type", "").lower()
        if ctype in ("figure", "table"):
            emb_dim = 896
        else:
            emb_dim = 384
        # Metadata with page number
        page_num = chunk["metadata"].get("page", None)
        node_metadata = {"min_page": page_num, "max_page": page_num}
        # Build the L0 node dict
        node = {
            "node_id": node_id,
            "paper_id": paper_id,
            "level": 0,
            "children_ids": [],         # no children for L0
            "text": content,
            "embedding_dim": emb_dim,
            "metadata": node_metadata
        }
        L0_nodes.append(node)
        raptor_nodes.append(node)
    # Optionally, sort chunks by page and position if needed (data is assumed in reading order)
    # L0_nodes.sort(key=lambda n: (n["metadata"]["min_page"], ))  # Further sorting by bbox if needed
    all_papers_L0[paper_id] = L0_nodes
    total_L0_count += len(L0_nodes)

print(f"Loaded {total_L0_count} L0 chunks from {len(all_papers_L0)} papers.")
# Estimate total groups for L1 and L2 summarization
total_L1_groups = 0
for pid, chunks in all_papers_L0.items():
    n = len(chunks)
    groups = (n + 7) // 8   # ceil(n/8)
    total_L1_groups += groups
total_L2_groups = 0
for pid, chunks in all_papers_L0.items():
    L1_count = (len(chunks) + 7) // 8
    L2_count = (L1_count + 5) // 6   # ceil(L1_count/6)
    total_L2_groups += L2_count
print(f"Planned L1 summary groups: {total_L1_groups}, Planned L2 summary groups: {total_L2_groups}")


Loading L0 chunks:   0%|          | 0/1023 [00:00<?, ?it/s]

Loaded 75431 L0 chunks from 1023 papers.
Planned L1 summary groups: 9865, Planned L2 summary groups: 2054


### Building Level-1 (L1) Summary Nodes

In this section, we construct the first layer of the RAPTOR hierarchy by summarizing groups of L0 chunks for every paper.

**What happens in this step:**

- **Grouping strategy**
  - L0 chunks are grouped in reading order.
  - Default group size: **8 chunks per L1 summary**.
  - Each group becomes a single L1 node.

- **Summarization with GPT-4o-mini**
  - All text from the group’s L0 chunks is concatenated.
  - A concise summary is generated using the OpenAI API.
  - Summaries are short, neutral, and capture the key idea of that span.

- **L1 node construction**
  - Node ID: `paper_id_L1_<group_index>`
  - `children_ids` contain the referenced L0 node IDs.
  - Page span is computed from the group’s children (`min_page`, `max_page`).
  - Embedding dimension is set to **384** (text-based).

- **Helper functions**
  - `summarize_group()` → handles API calls and text aggregation.
  - `build_level()` → generic grouping + summarization pipeline for any hierarchy level.

- **Progress tracking**
  - A tqdm progress bar reports L1 generation over all papers.
  - All L1 nodes are stored both per-paper (`all_papers_L1`) and in a flat list for embedding.

- **Sanity check**
  - After generation, we print a sample L1 node with page span + partial summary.

This stage transforms raw L0 chunks into structured, higher-level summaries that will form the mid-tier of the RAPTOR hierarchy.


In [10]:
from openai import OpenAI
client = OpenAI()

def summarize_group(chunk_texts):
    if not chunk_texts:
        return ""

    content = " ".join(chunk_texts)
    prompt = f"Summarize the following excerpt from a research paper:\n\"\"\"\n{content}\n\"\"\""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=150,
            temperature=0.2
        )
        summary = response.choices[0].message.content.strip()
        return summary

    except Exception as e:
        raise e


def build_level(prev_nodes, level, group_size, paper_id):
    """Group nodes from previous level and summarize to build the next level of hierarchy."""
    new_nodes = []
    n = len(prev_nodes)
    num_groups = (n + group_size - 1) // group_size  # ceil division
    for gi in range(num_groups):
        start = gi * group_size
        end = min(n, (gi + 1) * group_size)
        group_children = prev_nodes[start:end]
        # Gather text from children (some chunks could be non-text, but they have captions or content field)
        child_texts = [child["text"] for child in group_children if child.get("text")]
        summary_text = summarize_group(child_texts)
        node_id = f"{paper_id}_L{level}_{gi}"
        children_ids = [child["node_id"] for child in group_children]
        # Compute page span from children
        min_page = min(child["metadata"]["min_page"] for child in group_children)
        max_page = max(child["metadata"]["max_page"] for child in group_children)
        new_node = {
            "node_id": node_id,
            "paper_id": paper_id,
            "level": level,
            "children_ids": children_ids,
            "text": summary_text,
            "embedding_dim": 384,  # summary is text-based
            "metadata": {"min_page": min_page, "max_page": max_page}
        }
        new_nodes.append(new_node)
        raptor_nodes.append(new_node)
    return new_nodes

# Generate L1 nodes for all papers
all_papers_L1 = {}        # paper_id -> list of L1 nodes
L1_nodes_flat = []        # flat list of all L1 nodes (for embedding)
pbar = tqdm(total=total_L1_groups, desc="Generating L1 summaries")
for paper_id, L0_nodes in all_papers_L0.items():
    # Build L1 for this paper
    L1_nodes = build_level(L0_nodes, level=1, group_size=8, paper_id=paper_id)
    all_papers_L1[paper_id] = L1_nodes
    L1_nodes_flat.extend(L1_nodes)
    # Update progress by the number of groups processed for this paper
    pbar.update(len(L1_nodes))
pbar.close()

print(f"Generated {len(L1_nodes_flat)} L1 summary nodes in total.")
# Quick sanity check: report a sample L1 summary
if L1_nodes_flat:
    sample = L1_nodes_flat[0]
    print(f"Sample L1 node: {sample['node_id']} spans pages {sample['metadata']['min_page']}-{sample['metadata']['max_page']}\nSummary: {sample['text'][:100]}...")


Generating L1 summaries:   0%|          | 0/9865 [00:00<?, ?it/s]

Generated 9865 L1 summary nodes in total.
Sample L1 node: 1904.07640v1_L1_0 spans pages 2-8
Summary: The research paper addresses the challenges of post-market surveillance for medical devices, particu...


In [6]:
import os
print(os.getenv("OPENAI_API_KEY") is not None)


True


In [7]:
print("Total L1 groups:", total_L1_groups)
print("Total L2 groups:", total_L2_groups)
print("Estimated GPT calls:", total_L1_groups + total_L2_groups)


Total L1 groups: 9865
Total L2 groups: 2054
Estimated GPT calls: 11919


### Building Level-2 (L2) Summary Nodes

In this stage, we move one level higher in the RAPTOR hierarchy by summarizing groups of L1 nodes. L2 summaries provide a broad, high-level view of each paper.

**What happens in this step:**

- **Grouping strategy**
  - L1 nodes are grouped in their original order.
  - Default size: **6 L1 nodes per L2 summary**  
    (≈ up to 48 L0 chunks per L2 node).

- **Summarization**
  - For each group, we concatenate the L1 texts.
  - GPT-4o-mini generates a concise, top-level summary.
  - Many papers end up with only 1–2 L2 summaries.

- **L2 node creation**
  - Node ID: `paper_id_L2_<index>`
  - `children_ids` hold references to the group’s L1 nodes.
  - Metadata includes the page span covered by the children.
  - Embedding dimension fixed at **384** for text summaries.

- **Reuse of generic pipeline**
  - The same `build_level()` function handles grouping + summarization.
  - No separate summarization logic is needed for L2.

- **Progress tracking**
  - tqdm progress bar reflects total number of L2 groups across all papers.
  - All generated nodes stored in:
    - `all_papers_L2` (per-paper lists)
    - `L2_nodes_flat` (global flat list for embeddings and indexing)

- **Sanity check**
  - After generation, we print one L2 node showing the page span and the first portion of its summary.

At this point, the full RAPTOR hierarchy is complete:
- **L0:** raw chunks  
- **L1:** mid-level segment summaries  
- **L2:** high-level paper summaries  


In [11]:
all_papers_L2 = {}    # paper_id -> list of L2 nodes
L2_nodes_flat = []    # flat list of all L2 nodes
total_L2_count = 0

# Compute total L2 groups from actual L1 counts (for progress bar)
total_L2_groups = 0
for paper_id, L1_nodes in all_papers_L1.items():
    groups = (len(L1_nodes) + 5) // 6   # ceil division for L2 groups
    total_L2_groups += groups

pbar2 = tqdm(total=total_L2_groups, desc="Generating L2 summaries")
for paper_id, L1_nodes in all_papers_L1.items():
    L2_nodes = build_level(L1_nodes, level=2, group_size=6, paper_id=paper_id)
    all_papers_L2[paper_id] = L2_nodes
    L2_nodes_flat.extend(L2_nodes)
    total_L2_count += len(L2_nodes)
    pbar2.update(len(L2_nodes))
pbar2.close()

print(f"Generated {total_L2_count} L2 summary nodes in total.")
if L2_nodes_flat:
    sample2 = L2_nodes_flat[0]
    print(f"Sample L2 node: {sample2['node_id']} spans pages {sample2['metadata']['min_page']}-{sample2['metadata']['max_page']}\nSummary: {sample2['text'][:100]}...")


Generating L2 summaries:   0%|          | 0/2054 [00:00<?, ?it/s]

Generated 2054 L2 summary nodes in total.
Sample L2 node: 1904.07640v1_L2_0 spans pages 2-26
Summary: The research paper explores the challenges of post-market surveillance for medical devices, specific...


### Computing and Saving Embeddings for L1 and L2 Nodes

With the L1 and L2 summaries ready, we now compute their vector embeddings and store them for later retrieval and indexing.

**Key points:**

- **Embedding model:** MiniLM (384-dim vectors)
- **What we embed:**  
  - All L1 summaries → `L1_embedding_array`  
  - All L2 summaries → `L2_embedding_array`
- **Normalization:**  
  - `normalize_embeddings=True` ensures unit-length vectors (cosine-ready)
- **Storage format:**  
  Each pickle file contains a list of dictionaries with:
  - `id` — node identifier  
  - `text` — the summary text  
  - `embedding` — NumPy vector (384,)  
- **Output files:**  
  - `all_papers_L1_raptor_text.pkl`  
  - `all_papers_L2_raptor_text.pkl`
- **Directories:** Created under `data2/RAG/Version_V2/embeddings/`

These embeddings will be used in the next step to build FAISS indexes for hierarchical retrieval.


In [12]:
# Prepare lists of texts for embedding
L1_texts = [node["text"] for node in L1_nodes_flat]
L2_texts = [node["text"] for node in L2_nodes_flat]

# Compute 384-d embeddings for all L1 and L2 texts (in batches, if needed)
# Note: SentenceTransformer.encode will handle batching internally.
print("Computing embeddings for all L1 summaries...")
L1_embedding_array = embed_model.encode(L1_texts, batch_size=128, normalize_embeddings=True)
print("Computing embeddings for all L2 summaries...")
L2_embedding_array = embed_model.encode(L2_texts, batch_size=128, normalize_embeddings=True)

# Build embedding records (list of dicts) for pickle
L1_embedding_records = []
for node, vec in zip(L1_nodes_flat, L1_embedding_array):
    L1_embedding_records.append({
        "id": node["node_id"],
        "text": node["text"],
        "embedding": vec  # this is a numpy array of shape (384,)
    })
L2_embedding_records = []
for node, vec in zip(L2_nodes_flat, L2_embedding_array):
    L2_embedding_records.append({
        "id": node["node_id"],
        "text": node["text"],
        "embedding": vec
    })

# Create output directories if they don't exist
os.makedirs("../data2/RAG/Version_V2/raptor", exist_ok=True)
os.makedirs("../data2/RAG/Version_V2/embeddings", exist_ok=True)

# Save the embeddings to pickle files
with open("../data2/RAG/Version_V2/embeddings/all_papers_L1_raptor_text.pkl", "wb") as f:
    pickle.dump(L1_embedding_records, f)
with open("../data2/RAG/Version_V2/embeddings/all_papers_L2_raptor_text.pkl", "wb") as f:
    pickle.dump(L2_embedding_records, f)

print(f"Saved L1 embeddings for {len(L1_embedding_records)} nodes to 'all_papers_L1_raptor_text.pkl'.")
print(f"Saved L2 embeddings for {len(L2_embedding_records)} nodes to 'all_papers_L2_raptor_text.pkl'.")


Computing embeddings for all L1 summaries...
Computing embeddings for all L2 summaries...
Saved L1 embeddings for 9865 nodes to 'all_papers_L1_raptor_text.pkl'.
Saved L2 embeddings for 2054 nodes to 'all_papers_L2_raptor_text.pkl'.


### Saving All Hierarchical Nodes to a JSONL File

At this point, we have generated every node in the hierarchy (L0, L1, L2). In this step, we export them into a single JSONL file so the full structure can be inspected or reconstructed later without re-running summarization.

**What we save for each node:**
- `node_id` — unique identifier  
- `paper_id` — origin paper  
- `level` — 0 (L0), 1 (L1), or 2 (L2)  
- `children_ids` — child node references (empty for L0)  
- `text` — chunk text or summary  
- `embedding_dim` — 384 or 896 (for reference)  
- `metadata` — `min_page` and `max_page`  

All nodes are written line-by-line to `raptor_nodes.jsonl`, making the dataset easy to stream or partially load.


In [13]:
# Save all nodes (L0+L1+L2) to a JSONL file
out_path = "../data2/RAG/Version_V2/raptor/raptor_nodes.jsonl"
with open(out_path, "w") as f:
    for node in raptor_nodes:
        json.dump(node, f)
        f.write("\n")
print(f"Saved {len(raptor_nodes)} nodes (L0, L1, L2) to '{out_path}'.")


Saved 87350 nodes (L0, L1, L2) to '../data2/RAG/Version_V2/raptor/raptor_nodes.jsonl'.


### Building FAISS Indexes for L1 and L2 Embeddings

In this step, we create FAISS indexes for the new summary embeddings so we can perform fast cosine-similarity search over L1 and L2 nodes.

**Key points:**

- **Embedding prep**
  - Convert all L1 and L2 embeddings into NumPy float32 matrices.
  - Vectors are already normalized → inner product = cosine similarity.

- **FAISS index type**
  - We use `IndexFlatIP(384)` for both L1 and L2.
  - This is a simple, efficient flat index suitable for medium-sized datasets.

- **Index construction**
  - `index_L1.add(L1_matrix)`  
  - `index_L2.add(L2_matrix)`

- **Saved output files**
  - `text_L1_raptor.index.faiss`
  - `text_L2_raptor.index.faiss`

These indexes allow fast retrieval of mid-level (L1) and high-level (L2) summaries without touching the original L0 chunk index.


In [14]:
import faiss

# Convert embedding lists to numpy arrays for Faiss
L1_matrix = np.stack([rec["embedding"] for rec in L1_embedding_records]).astype('float32')
L2_matrix = np.stack([rec["embedding"] for rec in L2_embedding_records]).astype('float32')

# Build FAISS indexes (Inner Product for cosine similarity on normalized vectors)
index_L1 = faiss.IndexFlatIP(384)
index_L2 = faiss.IndexFlatIP(384)
index_L1.add(L1_matrix)
index_L2.add(L2_matrix)

# Save the indexes to files
os.makedirs("../data2/RAG/Version_V2/indexes", exist_ok=True)
faiss.write_index(index_L1, "../data2/RAG/Version_V2/indexes/text_L1_raptor.index.faiss")
faiss.write_index(index_L2, "../data2/RAG/Version_V2/indexes/text_L2_raptor.index.faiss")

print(f"L1 index saved with {index_L1.ntotal} vectors of dimension {index_L1.d}.")
print(f"L2 index saved with {index_L2.ntotal} vectors of dimension {index_L2.d}.")


L1 index saved with 9865 vectors of dimension 384.
L2 index saved with 2054 vectors of dimension 384.


### Converting Pickled Embeddings to JSON

This step converts the saved pickle files for L1 and L2 embeddings into JSON format for easier inspection and interoperability.

**What this code does:**
- Loads each `.pkl` file containing embedding records.
- Converts NumPy embedding vectors into Python lists (so JSON can serialize them).
- Writes the result to:
  - `all_papers_L1_raptor_text.json`
  - `all_papers_L2_raptor_text.json`
- Uses a small helper function `pkl_to_json()` to keep the process clean and reusable.

These JSON files are useful when previewing the hierarchy or debugging the embedding contents without loading NumPy arrays.


In [15]:
import pickle, json

def pkl_to_json(pkl_path, json_path):
    # Load the pickle file
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    # Convert embeddings (numpy arrays) to lists so JSON can handle them
    for item in data:
        if "embedding" in item:
            item["embedding"] = item["embedding"].tolist()

    # Save JSON
    with open(json_path, "w") as f:
        json.dump(data, f, indent=2)

    print(f"Saved JSON to: {json_path}")


# Convert L1 summaries
pkl_to_json(
    "../data2/RAG/Version_V2/embeddings/all_papers_L1_raptor_text.pkl",
    "../data2/RAG/Version_V2/embeddings/all_papers_L1_raptor_text.json"
)

# Convert L2 summaries
pkl_to_json(
    "../data2/RAG/Version_V2/embeddings/all_papers_L2_raptor_text.pkl",
    "../data2/RAG/Version_V2/embeddings/all_papers_L2_raptor_text.json"
)


Saved JSON to: ../data2/RAG/Version_V2/embeddings/all_papers_L1_raptor_text.json
Saved JSON to: ../data2/RAG/Version_V2/embeddings/all_papers_L2_raptor_text.json


### Inspecting the Hierarchy for a Single Paper


In [18]:
import json

paper_id = "2312.11514"
path = "../data2/RAG/Version_V2/raptor/raptor_nodes.jsonl"

L0 = []
L1 = []
L2 = []

with open(path, "r") as f:
    for line in f:
        node = json.loads(line)
        if node["paper_id"] == paper_id:
            if node["level"] == 0:
                L0.append(node)
            elif node["level"] == 1:
                L1.append(node)
            elif node["level"] == 2:
                L2.append(node)

print("Counts:")
print("L0:", len(L0))
print("L1:", len(L1))
print("L2:", len(L2))


Counts:
L0: 109
L1: 14
L2: 3


In [19]:
def print_summaries(nodes, title):
    print(f"\n=== {title} ===")
    for n in nodes:
        print(f"\n--- {n['node_id']} (pages {n['metadata']['min_page']}-{n['metadata']['max_page']}) ---")
        print(n["text"])
        
print_summaries(L1, "L1 Summaries")
print_summaries(L2, "L2 Summaries")



=== L1 Summaries ===

--- 2312.11514_L1_0 (pages 1-2) ---
The excerpt discusses advancements in large language models (LLMs) and proposes a method to improve inference efficiency by utilizing flash memory for parameter storage. It highlights the limitations of traditional DRAM in terms of bandwidth and capacity, suggesting that flash memory, despite its lower performance, can be leveraged to store larger models. The authors introduce techniques to selectively load model parameters on demand, which can significantly reduce inference latency and allow for the execution of models larger than the available DRAM. Their findings indicate that these optimizations can lead to up to 20 times faster inference compared to naive implementations on various hardware backends. The paper emphasizes the importance of understanding hardware characteristics and challenges in designing efficient algorithms for LLM inference, particularly when using flash memory, which performs better with

--- 2312.11514